# MAE 6246 - Week 1 Python Example

## Nonlinear pendulum and local linearizations

This notebook accompanies the Week 1 note on state-space models and linearization. We will:

1. simulate the nonlinear torque-driven pendulum;
2. construct Jacobian linearizations about the downward and upright equilibria;
3. compare each local model with the nonlinear response in time and in the phase plane; and
4. quantify how approximation error changes with the initial perturbation.

The angle convention is $\theta=0$ downward and $\theta=\pi$ upright. All simulations below use zero applied torque.

## 1. Model

For a point mass $m$ on a massless rod of length $\ell$,

$$m\ell^2\ddot\theta=-mg\ell\sin\theta+u.$$

With $x=[\theta,\dot\theta]^T$,

$$\dot x=f(x,u)=\begin{bmatrix}x_2\\-(g/\ell)\sin x_1+u/(m\ell^2)\end{bmatrix}.$$

At the equilibrium $x^\star=[\theta_{eq},0]^T$, the Jacobian is

$$A(\theta_{eq})=\begin{bmatrix}0&1\\-(g/\ell)\cos\theta_{eq}&0\end{bmatrix}.$$

The linear state is the perturbation $\delta x=x-x^\star$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.dpi": 120, "axes.grid": True})

g = 9.81       # gravitational acceleration [m/s^2]
ell = 1.0      # pendulum length [m]
m = 1.0        # point mass [kg]

In [ ]:
def pendulum_rhs(t, x, torque=lambda t: 0.0):
    """Nonlinear state equation with x = [theta, theta_dot]."""
    theta, theta_dot = x
    u = torque(t)
    return np.array([
        theta_dot,
        -(g / ell) * np.sin(theta) + u / (m * ell**2),
    ])


def linear_matrix(theta_eq):
    """Return the Jacobian A evaluated at [theta_eq, 0]."""
    return np.array([
        [0.0, 1.0],
        [-(g / ell) * np.cos(theta_eq), 0.0],
    ])


def compare_models(theta_eq, delta_theta0, t_final=6.0):
    """Simulate the nonlinear and local perturbation models."""
    t_eval = np.linspace(0.0, t_final, 1001)

    # The nonlinear model uses the physical angle theta.
    x0 = np.array([theta_eq + delta_theta0, 0.0])
    nonlinear = solve_ivp(
        pendulum_rhs,
        (0.0, t_final),
        x0,
        t_eval=t_eval,
        rtol=1e-10,
        atol=1e-12,
    )

    # The linearized model uses delta_x = x - x_star.
    A = linear_matrix(theta_eq)
    delta_x0 = np.array([delta_theta0, 0.0])
    linear = solve_ivp(
        lambda t, delta_x: A @ delta_x,
        (0.0, t_final),
        delta_x0,
        t_eval=t_eval,
        rtol=1e-10,
        atol=1e-12,
    )

    theta_nonlinear = nonlinear.y[0]
    theta_linear = theta_eq + linear.y[0]
    return t_eval, theta_nonlinear, theta_linear

## 2. Downward equilibrium

At $\theta_{eq}=0$,

$$A_d=\begin{bmatrix}0&1\\-g/\ell&0\end{bmatrix}.$$

Compare initial displacements of 5, 30, and 90 degrees. Before running the cell, predict which response will remain closest to its linear approximation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)

for ax, angle_deg in zip(axes, [5, 30, 90]):
    t, theta_nl, theta_lin = compare_models(
        theta_eq=0.0,
        delta_theta0=np.deg2rad(angle_deg),
    )
    ax.plot(t, np.rad2deg(theta_nl), label="nonlinear", lw=2)
    ax.plot(t, np.rad2deg(theta_lin), "--", label="linearized", lw=2)
    ax.set_title(f"initial angle = {angle_deg} deg")
    ax.set_xlabel("time [s]")
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("angle [deg]")
axes[0].legend()
fig.suptitle("Linearization about the downward equilibrium")
fig.tight_layout()
plt.show()

### Interpretation

- At 5 degrees, the curves nearly coincide.
- At 30 degrees, a phase error becomes visible.
- At 90 degrees, the linear model is qualitatively suggestive but quantitatively poor.

The small-angle approximation has leading error

$$\sin\theta-\theta\approx-\frac{\theta^3}{6},$$

so a tenfold decrease in a sufficiently small angle should reduce the instantaneous approximation error by roughly a factor of $10^3$.

## 3. Quantify the local-model error

For each initial angle, compute the maximum angle difference over three seconds. A log-log plot makes the small-perturbation scaling easier to see.

In [ ]:
initial_angles_deg = np.geomspace(0.1, 60.0, 30)
max_errors_deg = []

for angle_deg in initial_angles_deg:
    t, theta_nl, theta_lin = compare_models(
        theta_eq=0.0,
        delta_theta0=np.deg2rad(angle_deg),
        t_final=3.0,
    )
    error = np.max(np.abs(theta_nl - theta_lin))
    max_errors_deg.append(np.rad2deg(error))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(initial_angles_deg, max_errors_deg, "o-")
ax.set_xlabel("initial angle [deg]")
ax.set_ylabel("maximum angle error [deg]")
ax.set_title("Error of the downward linearization over 3 seconds")
ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

There is no universal angle at which the linearization suddenly becomes invalid. Model validity depends on the acceptable error, time interval, input, and question being asked.

## 4. Upright equilibrium

At $\theta_{eq}=\pi$,

$$A_u=\begin{bmatrix}0&1\\+g/\ell&0\end{bmatrix}.$$

The plotted quantity is the perturbation $\delta\theta=\theta-\pi$, not the physical angle itself.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

for angle_deg in [0.5, 2.0, 8.0]:
    t, theta_nl, theta_lin = compare_models(
        theta_eq=np.pi,
        delta_theta0=np.deg2rad(angle_deg),
        t_final=1.5,
    )
    ax.plot(
        t,
        np.rad2deg(theta_nl - np.pi),
        label=f"nonlinear, {angle_deg:g} deg",
        lw=2,
    )
    ax.plot(
        t,
        np.rad2deg(theta_lin - np.pi),
        "--",
        label=f"linearized, {angle_deg:g} deg",
        lw=1.5,
    )

ax.set_xlabel("time [s]")
ax.set_ylabel(r"perturbation $\theta-\pi$ [deg]")
ax.set_title("Linearization about the upright equilibrium")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()
plt.show()

The nonlinear and local responses initially agree, but both move away from the upright equilibrium. Eventually, the local approximation fails because the state has left the neighborhood where the Jacobian was evaluated.

This experiment separates two ideas that will be formalized later:

- **local accuracy:** whether the linearized and nonlinear trajectories agree near the operating point;
- **stability:** whether trajectories that start near the operating point remain near it.

## 5. Phase-plane comparison

A time history shows *when* two responses separate. A phase portrait shows how the state trajectories differ geometrically. We plot the perturbation coordinates $(\delta\theta,\delta\dot\theta)$ so that each equilibrium appears at the origin.

Each panel contains an ensemble of initial conditions that varies both angle and angular velocity. We integrate forward and backward from every marked state to expose the surrounding orbit geometry. All trajectories use one color: solid lines are nonlinear solutions and dashed lines are linearized solutions.

In [ ]:
from matplotlib.lines import Line2D


def state_trajectories(theta_eq, delta_x0, t_final, samples=1001):
    """Trace nonlinear and linear phase curves in both time directions."""
    x_star = np.array([theta_eq, 0.0])

    A = linear_matrix(theta_eq)
    nonlinear_initial = x_star + np.asarray(delta_x0)
    linear_initial = np.asarray(delta_x0)

    def trace(rhs, initial_state):
        forward_times = np.linspace(0.0, t_final, samples)
        backward_times = np.linspace(0.0, -t_final, samples)
        options = dict(rtol=1e-10, atol=1e-12)

        forward = solve_ivp(
            rhs, (0.0, t_final), initial_state,
            t_eval=forward_times, **options
        )
        backward = solve_ivp(
            rhs, (0.0, -t_final), initial_state,
            t_eval=backward_times, **options
        )

        times = np.concatenate((backward.t[::-1], forward.t[1:]))
        states = np.concatenate(
            (backward.y[:, ::-1], forward.y[:, 1:]), axis=1
        )
        return times, states

    times, nonlinear = trace(pendulum_rhs, nonlinear_initial)
    _, linear = trace(lambda t, delta_x: A @ delta_x, linear_initial)
    delta_nonlinear = nonlinear - x_star[:, None]
    return times, delta_nonlinear, linear


def add_phase_comparison(ax, theta_eq, initial_states_deg, t_final, title):
    """Overlay nonlinear and linear phase trajectories on one axis."""
    trajectory_color = "tab:blue"

    for angle_deg, velocity_deg_s in initial_states_deg:
        delta_x0 = np.deg2rad([angle_deg, velocity_deg_s])
        _, delta_nl, delta_lin = state_trajectories(
            theta_eq, delta_x0, t_final
        )

        ax.plot(
            np.rad2deg(delta_nl[0]),
            np.rad2deg(delta_nl[1]),
            color=trajectory_color,
            lw=2.2,
        )
        ax.plot(
            np.rad2deg(delta_lin[0]),
            np.rad2deg(delta_lin[1]),
            color=trajectory_color,
            ls="--",
            lw=1.7,
        )
        ax.scatter(
            angle_deg, velocity_deg_s, s=32, facecolor="white",
            edgecolor=trajectory_color, linewidth=1.2, zorder=4
        )

    ax.scatter(0.0, 0.0, marker="*", s=95, color="black", zorder=5)
    ax.axhline(0.0, color="0.75", lw=0.8)
    ax.axvline(0.0, color="0.75", lw=0.8)
    ax.set_title(title)
    ax.set_xlabel(r"angle perturbation $\delta\theta$ [deg]")
    ax.set_ylabel(r"angular-velocity perturbation $\delta\dot\theta$ [deg/s]")
    ax.grid(True, alpha=0.25)


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

add_phase_comparison(
    axes[0],
    theta_eq=0.0,
    initial_states_deg=[
        (-80.0, -20.0), (-55.0, 65.0), (-30.0, -110.0),
        (-12.0, 35.0), (0.0, 140.0), (20.0, -70.0),
        (38.0, 105.0), (62.0, -45.0), (85.0, 25.0),
    ],
    t_final=1.8,
    title="Hanging equilibrium",
)
add_phase_comparison(
    axes[1],
    theta_eq=np.pi,
    initial_states_deg=[
        (-3.0, -8.0), (-3.0, 0.0), (-3.0, 8.0),
        (-1.0, -4.0), (-1.0, 4.0),
        (1.0, -4.0), (1.0, 4.0),
        (3.0, -8.0), (3.0, 0.0), (3.0, 8.0),
    ],
    t_final=0.9,
    title="Inverted equilibrium",
)
axes[1].set_xlim(-15.0, 15.0)
axes[1].set_ylim(-50.0, 50.0)

model_legend = [
    Line2D([0], [0], color="0.15", lw=2.2, label="nonlinear"),
    Line2D([0], [0], color="0.15", lw=1.7, ls="--", label="linearized"),
    Line2D([0], [0], marker="o", markerfacecolor="white",
           markeredgecolor="0.15", color="none", lw=0, label="initial state"),
    Line2D([0], [0], marker="*", color="black", lw=0, markersize=10, label="equilibrium"),
]
fig.legend(
    handles=model_legend,
    loc="upper center",
    ncol=4,
    frameon=False,
    bbox_to_anchor=(0.5, 0.94),
)
fig.suptitle("Nonlinear and linearized phase trajectories", y=0.99)
fig.tight_layout(rect=(0.0, 0.0, 1.0, 0.86))
plt.show()

### What to notice

- Near the hanging equilibrium, trajectories that remain close to the origin are almost indistinguishable from the ellipses predicted by the linear model. Larger nonlinear orbits are visibly non-elliptical.
- Near the inverted equilibrium, the nonlinear and linear trajectories initially share the same local direction. They separate once the pendulum leaves the neighborhood of the equilibrium.
- The inverted linear model continues along an unbounded saddle trajectory, whereas the nonlinear pendulum eventually passes through a hanging configuration.

The linearization is therefore a **local approximation to the vector field**, not a global replacement for the nonlinear phase portrait.

## 6. Student investigations

1. Choose a maximum acceptable angle error, such as 1 degree. Estimate the largest downward initial angle satisfying that tolerance over three seconds.
2. Change the pendulum length. Explain how the response time scale changes.
3. Add a constant torque. Determine the new equilibrium and modify the linearization accordingly.
4. Give the pendulum a nonzero initial angular velocity. Compare the region over which the local model remains accurate.
5. Explain why the upright linearization can be accurate initially even though the equilibrium is unstable.